In [1]:
import pandas as pd
import requests
import time
import os

/Users/tanvirege/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/tanvirege/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [7]:
# Load environment variables directly from .env file
from dotenv import load_dotenv
import os

# Automatically finds and loads key-value pairs from .env in workspace
load_dotenv(override=True)

True

In [8]:
# Google Maps API Setup - Automatically loaded from .env
import os
import requests
import json
from dotenv import load_dotenv

# Load .env file
load_dotenv(override=True)

GOOGLE_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY")

if not GOOGLE_API_KEY or GOOGLE_API_KEY == "your_google_maps_api_key_here":
    print("⚠️ GOOGLE_MAPS_API_KEY is not configured in .env file")
    print("   Please update your key in the .env file at the project root.")
else:
    print(f"✅ Google Maps API key loaded from .env (ends with: ...{GOOGLE_API_KEY[-4:]})")

def geocode_place(place_name, api_key=None):
    """Convert place name to lat/lon using Google Geocoding API."""
    api_key = api_key or os.environ.get("GOOGLE_MAPS_API_KEY")
    if not api_key or api_key == "your_google_maps_api_key_here":
        print("❌ No valid Google Maps API key set in .env")
        return None
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": place_name, "key": api_key}
    r = requests.get(url, params=params, timeout=10)
    data = r.json()
    if data["status"] == "OK" and data["results"]:
        loc = data["results"][0]["geometry"]["location"]
        return {
            "place_name": place_name,
            "formatted_address": data["results"][0]["formatted_address"],
            "lat": loc["lat"],
            "lng": loc["lng"],
            "place_id": data["results"][0]["place_id"]
        }
    print(f"❌ Geocoding failed: {data['status']}")
    return None


✅ Google Maps API key loaded from .env (ends with: ...HP3Y)


In [9]:
# Google Maps Directions API - Uses key loaded from .env
import os
import requests
import json
import re
from urllib.parse import unquote
from dotenv import load_dotenv

load_dotenv(override=True)

def get_directions(origin, destination, waypoints=None, api_key=None):
    """Get driving directions using Google Directions API."""
    api_key = api_key or os.environ.get("GOOGLE_MAPS_API_KEY")
    if not api_key or api_key == "your_google_maps_api_key_here":
        print("❌ No valid API key set in .env")
        return None
    
    url = "https://maps.googleapis.com/maps/api/directions/json"
    params = {
        "origin": origin,
        "destination": destination,
        "mode": "driving",
        "key": api_key
    }
    if waypoints:
        params["waypoints"] = "|".join([f"via:{wp}" for wp in waypoints])
    
    r = requests.get(url, params=params, timeout=15)
    data = r.json()
    
    if data["status"] != "OK" or not data["routes"]:
        print(f"❌ Directions failed: {data['status']}")
        if "error_message" in data:
            print(f"   {data['error_message']}")
        return None
    
    route = data["routes"][0]
    legs = []
    total_distance_m = 0
    total_duration_s = 0
    
    for i, leg in enumerate(route["legs"]):
        legs.append({
            "leg_index": i,
            "start_address": leg["start_address"],
            "end_address": leg["end_address"],
            "start_location": leg["start_location"],
            "end_location": leg["end_location"],
            "distance_m": leg["distance"]["value"],
            "distance_text": leg["distance"]["text"],
            "duration_s": leg["duration"]["value"],
            "duration_text": leg["duration"]["text"],
            "steps_count": len(leg["steps"]),
            "steps": [{"start_location": s.get("start_location"), "end_location": s.get("end_location")} for s in leg["steps"]]
        })
        total_distance_m += leg["distance"]["value"]
        total_duration_s += leg["duration"]["value"]
    
    return {
        "total_distance_km": round(total_distance_m / 1000, 2),
        "total_duration_hours": round(total_duration_s / 3600, 2),
        "total_distance_text": f"{total_distance_m/1000:.1f} km",
        "total_duration_text": f"{total_duration_s/3600:.1f} hours",
        "legs": legs,
        "overview_polyline": route["overview_polyline"]["points"]
    }

def parse_google_maps_url(gmaps_url, api_key=None):
    """Parse a long Google Maps URL to extract origin, destination, waypoints."""
    api_key = api_key or os.environ.get("GOOGLE_MAPS_API_KEY")
    if "/maps/dir/" not in gmaps_url:
        print(f"❌ Not a valid Google Maps directions URL")
        return None
    path_part = gmaps_url.split("/maps/dir/")[1].split("/@")[0].split("/data=")[0]
    raw_places = [p for p in path_part.split("/") if p]
    places = [unquote(p.replace("+", " ")) for p in raw_places]
    if len(places) < 2:
        print(f"❌ Need at least origin and destination, got: {places}")
        return None
    return {
        "origin": places[0],
        "destination": places[-1],
        "waypoints": places[1:-1] if len(places) > 2 else None,
        "raw_places": places
    }


In [30]:
# Example usage with a long URL (only runs if key is set):
long_url = "https://www.google.com/maps/dir/Egilsstaðir,+700,+Iceland/Mývatn,+660,+Iceland/@65.4462439,-17.0877355,7.6z/data=!4m14!4m13!1m5!1m1!1s0x48cc04800da89ed3:0x95bf416b2c7c7f54!2m2!1d-14.3994394!2d65.2609232!1m5!1m1!1s0x48cd9c44953c07dd:0xcde4cb0dbf732a88!2m2!1d-16.9961055!2d65.60386!3e0?entry=ttu&g_ep=EgoyMDI2MDcyOS4wIKXMDSoASAFQAw%3D%3D"
route_info = parse_google_maps_url(long_url)
if route_info:
    print(json.dumps(route_info, indent=2))
    directions = get_directions(
        origin=route_info["origin"],
        destination=route_info["destination"],
        waypoints=route_info["waypoints"]
    )
print(json.dumps(directions, indent=2))


{
  "origin": "Egilssta\u00f0ir, 700, Iceland",
  "destination": "M\u00fdvatn, 660, Iceland",
  "waypoints": null,
  "raw_places": [
    "Egilssta\u00f0ir, 700, Iceland",
    "M\u00fdvatn, 660, Iceland"
  ]
}
{
  "total_distance_km": 174.17,
  "total_duration_hours": 2.14,
  "total_distance_text": "174.2 km",
  "total_duration_text": "2.1 hours",
  "legs": [
    {
      "leg_index": 0,
      "start_address": "700 Egilssta\u00f0ir, Iceland",
      "end_address": "M\u00fdvatn, 660, Iceland",
      "start_location": {
        "lat": 65.26092779999999,
        "lng": -14.3994208
      },
      "end_location": {
        "lat": 65.615821,
        "lng": -17.0244046
      },
      "distance_m": 174168,
      "distance_text": "174 km",
      "duration_s": 7714,
      "duration_text": "2 hours 9 mins",
      "steps_count": 5,
      "steps": [
        {
          "start_location": {
            "lat": 65.26092779999999,
            "lng": -14.3994208
          },
          "end_location": {
    

In [31]:
# Dynamic Vedur Station Mapper with 15km Spatial Sampling
import os
import requests
import json
import math
import pandas as pd
from typing import Dict, List, Optional, Tuple, Any

VEDUR_BASE_URL = "https://api.vedur.is/weather"

VEDUR_COLUMN_MAP = {
    "station": "station_id",
    "name": "station_name",
    "time": "observation_time_utc",
    "year": "year",
    "month": "month",
    "day": "day",
    "hour": "hour_utc",
    "t": "air_temp_c",
    "tx": "air_temp_max_c",
    "tn": "air_temp_min_c",
    "rh": "relative_humidity_pct",
    "vp": "vapor_pressure_hpa",
    "td": "dew_point_c",
    "f": "wind_speed_avg_ms",
    "fx": "wind_speed_max_ms",
    "fg": "wind_gust_max_ms",
    "fgfx": "gust_factor",
    "d": "wind_dir_deg",
    "d_txt": "wind_dir_cardinal",
    "ps": "station_pressure_hpa",
    "p": "sea_level_pressure_hpa",
    "r": "precipitation_mm",
    "tg": "ground_temp_c",
    "t0": "road_surface_temp_c",
    "count_measurements": "measurement_count",
}

def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Calculate Great Circle distance in km between two lat/lon coordinates."""
    R = 6371.0
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = (math.sin(dlat / 2.0) ** 2 +
         math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) *
         math.sin(dlon / 2.0) ** 2)
    c = 2.0 * math.atan2(math.sqrt(a), math.sqrt(1.0 - a))
    return R * c

def decode_polyline(polyline_str: str) -> List[Tuple[float, float]]:
    """Decode Google Maps encoded polyline into list of (lat, lng) tuples."""
    index, lat, lng = 0, 0, 0
    coordinates = []
    changes = {'latitude': 0, 'longitude': 0}
    while index < len(polyline_str):
        for unit in ['latitude', 'longitude']:
            shift, result = 0, 0
            while True:
                byte = ord(polyline_str[index]) - 63
                index += 1
                result |= (byte & 0x1f) << shift
                shift += 5
                if not byte >= 0x20:
                    break
            if result & 1:
                changes[unit] = ~(result >> 1)
            else:
                changes[unit] = result >> 1
        lat += changes['latitude']
        lng += changes['longitude']
        coordinates.append((lat / 1e5, lng / 1e5))
    return coordinates

def get_active_vedur_stations(station_types=["sj", "sk"]):
    """Fetch active weather stations from Vedur API (/stations?active=true)."""
    url = f"{VEDUR_BASE_URL}/stations"
    params = {"active": "true"}
    try:
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        active_stations = [s for s in data if s.get("type") in station_types]
        print(f"✅ Loaded {len(active_stations)} active Vedur stations ({', '.join(station_types)})")
        return active_stations
    except Exception as e:
        print(f"❌ Error fetching stations: {e}")
        return []

def map_google_route_to_vedur_stations(directions_data: Dict[str, Any], max_distance_km: float = 30.0, sample_interval_km: float = 15.0):
    """
    Convert Google Maps route points into nearest active Vedur station IDs using 15km spatial sampling.
    """
    active_stations = get_active_vedur_stations()
    if not active_stations or not directions_data or "legs" not in directions_data:
        return []

    sample_points = []
    legs = directions_data.get("legs", [])
    
    # 1. Start location
    if legs and "start_location" in legs[0]:
        sample_points.append({
            "label": f"Origin: {legs[0].get('start_address', 'Start')}",
            "lat": legs[0]["start_location"]["lat"],
            "lng": legs[0]["start_location"]["lng"]
        })
    
    # 2. Intermediate step locations
    for leg_idx, leg in enumerate(legs):
        steps_data = leg.get("steps", [])
        if isinstance(steps_data, list):
            for step_idx, step in enumerate(steps_data):
                if isinstance(step, dict) and "end_location" in step:
                    end_loc = step["end_location"]
                    if isinstance(end_loc, dict) and "lat" in end_loc and "lng" in end_loc:
                        sample_points.append({
                            "label": f"Leg {leg_idx+1} Step {step_idx+1}",
                            "lat": end_loc["lat"],
                            "lng": end_loc["lng"]
                        })
    
    # 3. End location
    if legs and "end_location" in legs[-1]:
        sample_points.append({
            "label": f"Destination: {legs[-1].get('end_address', 'End')}",
            "lat": legs[-1]["end_location"]["lat"],
            "lng": legs[-1]["end_location"]["lng"]
        })

    # 4. Polyline spatial sampling at 15km intervals along road
    overview_polyline = directions_data.get("overview_polyline")
    if overview_polyline:
        poly_points = decode_polyline(overview_polyline)
        if poly_points:
            accumulated_dist = 0.0
            last_pt = poly_points[0]
            for pt in poly_points[1:]:
                dist = haversine_km(last_pt[0], last_pt[1], pt[0], pt[1])
                accumulated_dist += dist
                if accumulated_dist >= sample_interval_km:
                    sample_points.append({
                        "label": f"Polyline Sample (~{accumulated_dist:.1f}km)",
                        "lat": pt[0],
                        "lng": pt[1]
                    })
                    accumulated_dist = 0.0
                last_pt = pt

    # Match each point to nearest Vedur station
    matched_stations = []
    seen_ids = set()

    for pt in sample_points:
        closest = None
        min_dist = float("inf")
        for st in active_stations:
            st_lat, st_lon = st.get("lat"), st.get("lon")
            if st_lat is None or st_lon is None:
                continue
            dist = haversine_km(pt["lat"], pt["lng"], st_lat, st_lon)
            if dist < min_dist:
                min_dist = dist
                closest = st
        
        if closest and min_dist <= max_distance_km and closest["station"] not in seen_ids:
            seen_ids.add(closest["station"])
            matched_stations.append({
                "station_id": closest["station"],
                "station_name": closest.get("name"),
                "lat": closest.get("lat"),
                "lon": closest.get("lon"),
                "distance_km": round(min_dist, 2),
                "matched_point": pt["label"]
            })

    return matched_stations

def fetch_weather_for_station_ids(station_ids: List[int]):
    """Fetch latest hourly observation data from Vedur API for matched station IDs."""
    url = f"{VEDUR_BASE_URL}/observations/aws/hour/latest"
    all_obs = []
    for st_id in station_ids:
        try:
            r = requests.get(url, params={"station_id": st_id, "parameters": "all"}, timeout=10)
            r.raise_for_status()
            data = r.json()
            if data:
                all_obs.extend(data)
                print(f"✅ Station {st_id} ({data[0].get('name', '')}): {len(data)} record(s)")
        except Exception as e:
            print(f"❌ Station {st_id}: Error {e}")
    return pd.DataFrame(all_obs)


In [ ]:
# Vedur API column name -> full description mapping
VEDUR_COLUMN_MAP = {
    # Station metadata
    "station": "station_id",
    "name": "station_name",
    
    # Time dimensions
    "time": "observation_time_utc",
    "year": "year",
    "month": "month",
    "day": "day",
    "hour": "hour_utc",  # 1-24, where 24 = midnight
    
    # Temperature (Celsius)
    "t": "air_temp_c",           # Air temperature at observation time
    "tx": "air_temp_max_c",      # Maximum air temperature since last observation
    "tn": "air_temp_min_c",      # Minimum air temperature since last observation
    
    # Humidity & moisture
    "rh": "relative_humidity_pct",  # Relative humidity (%)
    "vp": "vapor_pressure_hpa",     # Vapor pressure (hPa)
    "td": "dew_point_c",            # Dew point temperature (C)
    
    # Wind
    "f": "wind_speed_avg_ms",       # Average wind speed (m/s) over 10-min period
    "fx": "wind_speed_max_ms",      # Maximum 10-min average wind speed (m/s)
    "fg": "wind_gust_max_ms",       # Maximum wind gust (m/s)
    "fgfx": "gust_factor",          # Ratio of max gust to max 10-min wind (fg/fx)
    "d": "wind_dir_deg",            # Wind direction (degrees, 0-360)
    "d_txt": "wind_dir_cardinal",   # Wind direction (cardinal: N, NE, E, etc.)
    "dsdev": "wind_dir_std_dev",    # Standard deviation of wind direction (degrees)
    
    # Pressure
    "ps": "station_pressure_hpa",   # Station level pressure (hPa)
    "p": "sea_level_pressure_hpa",  # Sea level pressure (hPa)
    
    # Precipitation
    "r": "precipitation_mm",        # Precipitation since last observation (mm)
    
    # Ground temperature
    "tg": "ground_temp_c",          # Ground temperature (C)
    "tgn": "ground_temp_min_c",     # Minimum ground temperature (C)
    
    # Road surface temperature (for road stations)
    "t0": "road_surface_temp_c",    # Road surface temperature (C)
    "t0x": "road_surface_temp_max_c",
    "t0n": "road_surface_temp_min_c",
    
    # Turf/grass temperature at various depths
    "tug5": "turf_temp_5cm_c",
    "tug10": "turf_temp_10cm_c",
    "tug15": "turf_temp_15cm_c",
    "tug20": "turf_temp_20cm_c",
    "tug50": "turf_temp_50cm_c",
    "tug100": "turf_temp_100cm_c",
    
    # Radiation
    "radgl": "global_radiation_wm2",       # Global radiation (W/m²)
    "radglx": "global_radiation_max_wm2",
    "radsc": "shortwave_radiation_wm2",    # Shortwave radiation (W/m²)
    "radscx": "shortwave_radiation_max_wm2",
    "radsws": "sunshine_duration_s",       # Sunshine duration (seconds)
    "radswsx": "sunshine_duration_max_s",
    "radlwi": "longwave_in_radiation_wm2", # Longwave incoming radiation (W/m²)
    "radlwix": "longwave_in_radiation_max_wm2",
    "radlws": "longwave_out_radiation_wm2", # Longwave outgoing radiation (W/m²)
    "radlwsx": "longwave_out_radiation_max_wm2",
    "raduv": "uv_radiation_wm2",           # UV radiation (W/m²)
    "raduvx": "uv_radiation_max_wm2",
    
    # Other
    "rsun": "sunshine_duration_min",       # Sunshine duration (minutes)
    "ts": "snow_depth_cm",                 # Snow depth (cm)
    "sal": "salinity_psu",                 # Salinity (PSU) - for marine stations
    "count_measurements": "measurement_count",  # Number of measurements in period
}

# Print nicely formatted
for short, full in VEDUR_COLUMN_MAP.items():
    print(f"  {short:>15} -> {full}")

In [ ]:
# Complete Pipeline: Route Weather Ingestion -> Parquet Export -> DuckDB Analytics
import os
import duckdb
import pandas as pd

# 1. Resolve Route Data (Google Maps or Fallback)
route_data = None
if "directions" in locals() and isinstance(directions, dict):
    route_data = directions
elif "route_info" in locals() and isinstance(route_info, dict):
    route_data = get_directions(route_info["origin"], route_info["destination"], route_info.get("waypoints"))

if not route_data:
    # Fallback sample directions if API key isn't set
    route_data = {
        "legs": [{
            "start_address": "Egilsstaðir, Iceland",
            "end_address": "Mývatn, Iceland",
            "start_location": {"lat": 65.260928, "lng": -14.3994208},
            "end_location": {"lat": 65.615821, "lng": -17.0244046},
            "steps": [
                {"end_location": {"lat": 65.3400, "lng": -15.1000}},
                {"end_location": {"lat": 65.5000, "lng": -16.0000}}
            ]
        }]
    }

# 2. Map Route to Weather Stations (15km spatial sampling)
matched_stations = map_google_route_to_vedur_stations(route_data, sample_interval_km=15.0)
print(f"✅ Matched {len(matched_stations)} weather stations along the route.")

# 3. Fetch Live Weather Data for ALL Matched Stations (Do NOT sample or discard!)
station_ids = [m["station_id"] for m in matched_stations]
df_weather_raw = fetch_weather_for_station_ids(station_ids)

# 4. Enrich & Rename Data for DuckDB Pipeline
if not df_weather_raw.empty:
    df_weather_renamed = df_weather_raw.rename(columns=VEDUR_COLUMN_MAP)
    
    df_stations_meta = pd.DataFrame(matched_stations)
    df_route_telemetry = pd.merge(
        df_weather_renamed,
        df_stations_meta[["station_id", "lat", "lon", "distance_km", "matched_point"]],
        on="station_id",
        how="left"
    )
    
    if "waypoint_id" not in df_route_telemetry.columns:
        df_route_telemetry["waypoint_id"] = range(1, len(df_route_telemetry) + 1)
    if "stop_name" not in df_route_telemetry.columns:
        df_route_telemetry["stop_name"] = df_route_telemetry["station_name"]
    if "region" not in df_route_telemetry.columns:
        df_route_telemetry["region"] = "North/East"
    if "latitude" not in df_route_telemetry.columns:
        df_route_telemetry["latitude"] = df_route_telemetry["lat"]
    if "longitude" not in df_route_telemetry.columns:
        df_route_telemetry["longitude"] = df_route_telemetry["lon"]
    if "timestamp" not in df_route_telemetry.columns:
        df_route_telemetry["timestamp"] = df_route_telemetry["observation_time_utc"]
    if "wind_speed_ms" not in df_route_telemetry.columns:
        df_route_telemetry["wind_speed_ms"] = df_route_telemetry.get("wind_speed_avg_ms", 5.0)
    if "wind_gust_ms" not in df_route_telemetry.columns:
        df_route_telemetry["wind_gust_ms"] = df_route_telemetry.get("wind_gust_max_ms", 8.0)
    if "temperature_c" not in df_route_telemetry.columns:
        df_route_telemetry["temperature_c"] = df_route_telemetry.get("air_temp_c", 10.0)
    if "precipitation_mm" not in df_route_telemetry.columns:
        df_route_telemetry["precipitation_mm"] = df_route_telemetry.get("precipitation_mm", 0.0)
    if "road_status" not in df_route_telemetry.columns:
        df_route_telemetry["road_status"] = df_route_telemetry["wind_gust_ms"].apply(
            lambda g: "Impassable" if g >= 20 else ("Caution Advised" if g >= 14 else "Open")
        )
    if "fuel_price_isk" not in df_route_telemetry.columns:
        df_route_telemetry["fuel_price_isk"] = 320.0
    if "fuel_brand" not in df_route_telemetry.columns:
        df_route_telemetry["fuel_brand"] = "N1"
    if "campsite_fee_isk" not in df_route_telemetry.columns:
        df_route_telemetry["campsite_fee_isk"] = 2500
    if "campsite_availability" not in df_route_telemetry.columns:
        df_route_telemetry["campsite_availability"] = "Available"
    if "daylight_hours" not in df_route_telemetry.columns:
        df_route_telemetry["daylight_hours"] = 16.0

    # 5. Save ALL Station Data to Parquet & CSV for DuckDB
    os.makedirs("data", exist_ok=True)
    parquet_path = "data/iceland_raw_telemetry.parquet"
    csv_path = "data/iceland_raw_telemetry.csv"
    
    df_route_telemetry.to_parquet(parquet_path, index=False)
    df_route_telemetry.to_csv(csv_path, index=False)
    print(f"💾 Saved all {len(df_route_telemetry)} station records to {parquet_path} and {csv_path}")

    # 6. Test DuckDB Query on Saved Parquet File
    conn = duckdb.connect()
    df_duckdb = conn.execute("""
        SELECT 
            waypoint_id,
            stop_name,
            latitude,
            longitude,
            wind_speed_ms,
            wind_gust_ms,
            temperature_c,
            road_status,
            CASE 
                WHEN wind_gust_ms >= 18 THEN 'CRITICAL: Rollover Risk'
                WHEN wind_gust_ms >= 14 THEN 'HIGH: Strong Crosswinds'
                WHEN wind_gust_ms >= 10 THEN 'MODERATE: Caution'
                ELSE 'CLEAR: Safe Driving'
            END AS hazard_alert
        FROM read_parquet('data/iceland_raw_telemetry.parquet')
        ORDER BY waypoint_id
    """).df()
    
    print("\n--- DUCKDB QUERY RESULTS (FOR STREAMLIT) ---")
    print(df_duckdb.to_string(index=False))


✅ Loaded 309 active Vedur stations (sj, sk)
✅ Matched 8 weather stations along the route.
✅ Station 4271 (Egilsstaðaflugvöllur): 1 record(s)
✅ Station 4300 (Mývatn): 1 record(s)
✅ Station 34148 (Jökuldalur): 1 record(s)
✅ Station 34238 (Möðrudalsöræfi II): 1 record(s)
✅ Station 34326 (Biskupsháls): 1 record(s)
✅ Station 4323 (Grímsstaðir á Fjöllum): 1 record(s)
✅ Station 34413 (Mývatnsöræfi): 1 record(s)
✅ Station 4406 (Krafla): 1 record(s)
💾 Saved all 8 station records to data/iceland_raw_telemetry.parquet and data/iceland_raw_telemetry.csv

--- DUCKDB QUERY RESULTS (FOR STREAMLIT) ---
 waypoint_id             stop_name  latitude  longitude  wind_speed_ms  wind_gust_ms  temperature_c road_status        hazard_alert
           1  Egilsstaðaflugvöllur 65.276176 -14.404600            6.1           8.7           11.9        Open CLEAR: Safe Driving
           2                Mývatn 65.619331 -16.976839            1.7           2.3           12.8        Open CLEAR: Safe Driving
          